## Setups and Imports

In [24]:
# Standard Library Imports
import sys
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Add src directory to path
project_root = Path().resolve().parents[1]
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Fuzzy string matching
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

# Project utilities
from utils.helpers import load_orders, set_style, save_figure

set_style()

print("All imports successful!")
print(f"Project root: {project_root}")

All imports successful!
Project root: /Users/seleteakpotosu-nartey/Downloads/Data Stuff/Gamezone/gamezone-analytics


## Load Data

In [25]:
# Load the processed file from Notebook 2
# This file has IS_ANOMALY, FULFILMENT_DAYS, IS_ZERO_PRICE,
# IS_PRICE_OUTLIER and IS_CURRENCY_SUSPECT already added

processed_path = project_root / 'data' / 'processed'
input_file = processed_path / 'orders_with_price_flags.csv'

df_raw = pd.read_csv(input_file, parse_dates=['PURCHASE_TS', 'SHIP_TS'])
df = df_raw.copy()


print(f'Rows:    {len(df):,}')
print(f'Columns: {len(df.columns)}')
print(f'\nColumns: {list(df.columns)}')

Rows:    21,864
Columns: 20

Columns: ['USER_ID', 'ORDER_ID', 'PURCHASE_TS', 'SHIP_TS', 'PRODUCT_NAME', 'PRODUCT_ID', 'USD_PRICE', 'PURCHASE_PLATFORM', 'MARKETING_CHANNEL', 'ACCOUNT_CREATION_METHOD', 'COUNTRY_CODE', 'REGION', 'FULFILMENT_DAYS', 'IS_ANOMALY', 'YEAR_MONTH', 'PURCHASE_YEAR', 'PURCHASE_MONTH', 'PURCHASE_YEARMONTH', 'IS_PRICE_OUTLIER', 'IS_ZERO_PRICE']


In [26]:
# PURCHASE_TS should be datetime, but we have some weird values in there that are causing parsing issues. 
# Let's coerce them to datetime and see what we get.
# Simply coerce — NaN stays NaN, everything else becomes datetime

df['PURCHASE_TS'] = pd.to_datetime(df['PURCHASE_TS'], errors='coerce')

## Fix Product Names

In [27]:
# Get all unique product names and their counts

product_counts = df['PRODUCT_NAME'].value_counts()
for name, count in product_counts.items():
    print(f"{count:>6,} - {name}")
    
print(f"\nTotal unique product names: {df['PRODUCT_NAME'].nunique()}")
print(f'Expected unique products: 9(based on domain knowledge)')

10,386 - Nintendo Switch
 4,662 - 27in 4K gaming monitor
 4,296 - JBL Quantum 100 Gaming Headset
   977 - Sony PlayStation 5 Bundle
   719 - Dell Gaming Mouse
   669 - Lenovo IdeaPad Gaming 3
    87 - Acer Nitro V Gaming Laptop
    61 - 27inches 4k gaming monitor
     7 - Razer Pro Gaming Headset

Total unique product names: 9
Expected unique products: 9(based on domain knowledge)


In [28]:
# Fuzzy matching to find similar product names

product_names = df['PRODUCT_NAME'].unique().tolist()

print("\nFuzzy matching results:")
print('Scores above 80 indicate strong similarity and likely duplicates:')

matches = []
for i, name in enumerate(product_names):
    for other_name in product_names[i+1:]:
        score = fuzz.token_sort_ratio(name, other_name)
        matches.append({
            'Name1': name,
            'Name2': other_name,
            'Score': score
        })
        
matches_df = pd.DataFrame(matches).sort_values(by='Score', ascending=False)

# Matches with score above 60, which may indicate potential duplicates or similar products

similar_products = matches_df[matches_df['Score'] > 60]
print(similar_products.to_string(index=False))


Fuzzy matching results:
Scores above 80 indicate strong similarity and likely duplicates:
                         Name1                      Name2  Score
        27in 4K gaming monitor 27inches 4k gaming monitor     92
JBL Quantum 100 Gaming Headset   Razer Pro Gaming Headset     63


In [29]:
# Confirm duplicate pair in detail
# Determine when each product name was used in the orders 
# and whether one is an old name and the other a new name

duplicate_names = ['27in 4K gaming monitor', '27inches 4k gaming monitor']

for name in duplicate_names:
    subset = df[df['PRODUCT_NAME'] == name]
    print(f"\nOrders with product name: '{name}'")
    print(f' Order count:      {len(subset):,}')
    print(f' Date range:       {subset["PURCHASE_TS"].dropna().min()} to {subset["PURCHASE_TS"].dropna().max()}')
    print(f' Unique order IDs: {subset["PRODUCT_ID"].nunique():,}')
    print(f' Product IDs used: {sorted(subset["PRODUCT_ID"].unique().tolist())}')
    print(f' Median price:     ${subset["USD_PRICE"].median():,.2f}')
    print(f' Price range:      ${subset["USD_PRICE"].min():,.2f} to ${subset["USD_PRICE"].max():,.2f}')


Orders with product name: '27in 4K gaming monitor'
 Order count:      4,662
 Date range:       2019-01-02 00:00:00 to 2021-02-26 00:00:00
 Unique order IDs: 7
 Product IDs used: ['1238', '7599', '7f86', '8364', '891b', 'e7e6', 'f443']
 Median price:     $438.17
 Price range:      $0.00 to $584.09

Orders with product name: '27inches 4k gaming monitor'
 Order count:      61
 Date range:       2019-01-01 00:00:00 to 2020-12-24 00:00:00
 Unique order IDs: 3
 Product IDs used: ['2a50', 'ab5d', 'afbf']
 Median price:     $359.98
 Price range:      $204.88 to $366.57


## Product ID Problem

In [30]:
# How many unique product IDs does each product name have?

print('Product ID counts per product name:')
print('A well governed dataset should have a 1:1 relationship between product name and product ID\n')

id_check = df.groupby('PRODUCT_NAME').agg(
    order_count=('ORDER_ID', 'count'),
    unique_ids=('PRODUCT_ID', 'nunique'),
    id_list=('PRODUCT_ID', lambda x: sorted(x.unique().tolist()))
).sort_values(by='unique_ids', ascending=False)

print(id_check[['order_count', 'unique_ids']].to_string())
print(f'\nTotal unique product names: {df["PRODUCT_NAME"].nunique()}')

Product ID counts per product name:
A well governed dataset should have a 1:1 relationship between product name and product ID

                                order_count  unique_ids
PRODUCT_NAME                                           
Nintendo Switch                       10386          12
27in 4K gaming monitor                 4662           7
Dell Gaming Mouse                       719           7
JBL Quantum 100 Gaming Headset         4296           7
Sony PlayStation 5 Bundle               977           4
27inches 4k gaming monitor               61           3
Lenovo IdeaPad Gaming 3                 669           3
Razer Pro Gaming Headset                  7           2
Acer Nitro V Gaming Laptop               87           1

Total unique product names: 9


In [31]:
# For each product, show which product IDs it uses and how many
# orders each ID has — this tells us which ID is the primary one
print('Product ID usage breakdown')
print('(Most frequent ID per product = likely the canonical one)\n')

for product in df['PRODUCT_NAME'].unique():
    subset = df[df['PRODUCT_NAME'] == product]
    id_counts = subset['PRODUCT_ID'].value_counts()
    print(f'{product}  ({len(subset):,} orders)')
    for pid, count in id_counts.items():
        pct = count / len(subset) * 100
        bar = '█' * int(pct / 2)
        print(f'  {pid}  {count:>5,} orders ({pct:>5.1f}%)  {bar}')
    print()

Product ID usage breakdown
(Most frequent ID per product = likely the canonical one)

Nintendo Switch  (10,386 orders)
  8d0d  6,024 orders ( 58.0%)  █████████████████████████████
  e682  3,806 orders ( 36.6%)  ██████████████████
  8e5d    383 orders (  3.7%)  █
  b5f7    114 orders (  1.1%)  
  0d23     18 orders (  0.2%)  
  7d63     12 orders (  0.1%)  
  6b8d     12 orders (  0.1%)  
  97c6      6 orders (  0.1%)  
  604c      5 orders (  0.0%)  
  da12      3 orders (  0.0%)  
  24c1      2 orders (  0.0%)  
  03ca      1 orders (  0.0%)  

Sony PlayStation 5 Bundle  (977 orders)
  54ed    970 orders ( 99.3%)  █████████████████████████████████████████████████
  df85      3 orders (  0.3%)  
  e22d      3 orders (  0.3%)  
  12b1      1 orders (  0.1%)  

27in 4K gaming monitor  (4,662 orders)
  891b  3,545 orders ( 76.0%)  ██████████████████████████████████████
  e7e6  1,102 orders ( 23.6%)  ███████████
  1238      7 orders (  0.2%)  
  7f86      5 orders (  0.1%)  
  7599      1 

## Mapping Tables for product name and product IDs

In [32]:
# Name mapping table
# Maps every known product name to its canonical version
# '27in 4K gaming monitor' is chosen as canonical because:
#   - It has 4,662 orders vs 61 for the legacy name
#   - It uses the standard abbreviated format consistent with other product names

name_mapping = {
    '27in 4K gaming monitor':          '27in 4K gaming monitor',      # canonical — no change
    '27inches 4k gaming monitor':      '27in 4K gaming monitor',      # legacy → canonical
    'Nintendo Switch':                 'Nintendo Switch',
    'Sony PlayStation 5 Bundle':       'Sony PlayStation 5 Bundle',
    'JBL Quantum 100 Gaming Headset':  'JBL Quantum 100 Gaming Headset',
    'Dell Gaming Mouse':               'Dell Gaming Mouse',
    'Acer Nitro V Gaming Laptop':      'Acer Nitro V Gaming Laptop',
    'Lenovo IdeaPad Gaming 3':         'Lenovo IdeaPad Gaming 3',
    'Razer Pro Gaming Headset':        'Razer Pro Gaming Headset',
}

# Convert to DataFrame for inspection and export
name_mapping_df = pd.DataFrame([
    {'original_name': k, 'canonical_name': v, 'is_duplicate': k != v}
    for k, v in name_mapping.items()
])

print('Product name mapping table')
print(name_mapping_df.to_string(index=False))

Product name mapping table
                 original_name                 canonical_name  is_duplicate
        27in 4K gaming monitor         27in 4K gaming monitor         False
    27inches 4k gaming monitor         27in 4K gaming monitor          True
               Nintendo Switch                Nintendo Switch         False
     Sony PlayStation 5 Bundle      Sony PlayStation 5 Bundle         False
JBL Quantum 100 Gaming Headset JBL Quantum 100 Gaming Headset         False
             Dell Gaming Mouse              Dell Gaming Mouse         False
    Acer Nitro V Gaming Laptop     Acer Nitro V Gaming Laptop         False
       Lenovo IdeaPad Gaming 3        Lenovo IdeaPad Gaming 3         False
      Razer Pro Gaming Headset       Razer Pro Gaming Headset         False


In [33]:
# Product ID mapping table
# For each product, we identify the canonical ID as the one with
# the most orders — the most frequently used ID is the current standard

print('Canonical ID = most frequently used ID per product\n')

# First apply the name fix so the duplicate monitor is treated as one product
df['PRODUCT_NAME_CLEAN'] = df['PRODUCT_NAME'].map(name_mapping)

id_mapping_rows = []
for product in sorted(df['PRODUCT_NAME_CLEAN'].unique()):
    subset = df[df['PRODUCT_NAME_CLEAN'] == product]
    id_counts = subset['PRODUCT_ID'].value_counts()

    # Most frequent ID is the canonical one
    canonical_id = id_counts.index[0]

    for pid in id_counts.index:
        id_mapping_rows.append({
            'product_name_clean': product,
            'original_product_id': pid,
            'canonical_product_id': canonical_id,
            'order_count': id_counts[pid],
            'is_canonical': pid == canonical_id
        })

id_mapping_df = pd.DataFrame(id_mapping_rows)

print(f'Total product IDs mapped: {len(id_mapping_df)}')
print(f'Unique canonical IDs:     {id_mapping_df["canonical_product_id"].nunique()}')
print()
print('Summary: canonical ID per product')
canonical_summary = id_mapping_df[id_mapping_df['is_canonical']][['product_name_clean', 'canonical_product_id', 'order_count']]
print(canonical_summary.to_string(index=False))

Canonical ID = most frequently used ID per product

Total product IDs mapped: 46
Unique canonical IDs:     8

Summary: canonical ID per product
            product_name_clean canonical_product_id  order_count
        27in 4K gaming monitor                 891b         3545
    Acer Nitro V Gaming Laptop                 22ea           87
             Dell Gaming Mouse                 8d4f          355
JBL Quantum 100 Gaming Headset                 ab0f         3446
       Lenovo IdeaPad Gaming 3                 04ac          433
               Nintendo Switch                 8d0d         6024
      Razer Pro Gaming Headset                 a6be            6
     Sony PlayStation 5 Bundle                 54ed          970


## Applying the mappings

In [34]:
# Build a lookup: original_product_id → canonical_product_id
id_lookup = dict(zip(
    id_mapping_df['original_product_id'],
    id_mapping_df['canonical_product_id']
))

# Apply both mappings to the working dataframe
# PRODUCT_NAME_CLEAN was already created above
# Now add PRODUCT_ID_CLEAN
df['PRODUCT_ID_CLEAN'] = df['PRODUCT_ID'].map(id_lookup)

# Verify the mappings worked
print('Verification: before vs after')
print()
print('Product names — before:')
print(f'  Unique names: {df["PRODUCT_NAME"].nunique()}')
print(f'  Names: {sorted(df["PRODUCT_NAME"].unique().tolist())}')
print()
print('Product names — after:')
print(f'  Unique names: {df["PRODUCT_NAME_CLEAN"].nunique()}')
print(f'  Names: {sorted(df["PRODUCT_NAME_CLEAN"].unique().tolist())}')
print()
print('Product IDs — before:')
print(f'  Unique IDs: {df["PRODUCT_ID"].nunique()}')
print()
print('Product IDs — after:')
print(f'  Unique IDs: {df["PRODUCT_ID_CLEAN"].nunique()}')

Verification: before vs after

Product names — before:
  Unique names: 9
  Names: ['27in 4K gaming monitor', '27inches 4k gaming monitor', 'Acer Nitro V Gaming Laptop', 'Dell Gaming Mouse', 'JBL Quantum 100 Gaming Headset', 'Lenovo IdeaPad Gaming 3', 'Nintendo Switch', 'Razer Pro Gaming Headset', 'Sony PlayStation 5 Bundle']

Product names — after:
  Unique names: 8
  Names: ['27in 4K gaming monitor', 'Acer Nitro V Gaming Laptop', 'Dell Gaming Mouse', 'JBL Quantum 100 Gaming Headset', 'Lenovo IdeaPad Gaming 3', 'Nintendo Switch', 'Razer Pro Gaming Headset', 'Sony PlayStation 5 Bundle']

Product IDs — before:
  Unique IDs: 46

Product IDs — after:
  Unique IDs: 8


In [35]:
# Confirm order counts are preserved after the name merge
# The two monitor variants should now combine into one row
print('Order counts per product after cleaning')
clean_counts = df['PRODUCT_NAME_CLEAN'].value_counts()
raw_counts   = df['PRODUCT_NAME'].value_counts()

print(f'Products before: {raw_counts.shape[0]}')
print(f'Products after:  {clean_counts.shape[0]}')
print()
for name, count in clean_counts.items():
    print(f'  {count:>6,} orders  →  {name}')
print()
print(f'Total orders: {clean_counts.sum():,} (should match original {len(df):,})')

Order counts per product after cleaning
Products before: 9
Products after:  8

  10,386 orders  →  Nintendo Switch
   4,723 orders  →  27in 4K gaming monitor
   4,296 orders  →  JBL Quantum 100 Gaming Headset
     977 orders  →  Sony PlayStation 5 Bundle
     719 orders  →  Dell Gaming Mouse
     669 orders  →  Lenovo IdeaPad Gaming 3
      87 orders  →  Acer Nitro V Gaming Laptop
       7 orders  →  Razer Pro Gaming Headset

Total orders: 21,864 (should match original 21,864)
